# Pruebas saber 11

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MiApp") \
    .getOrCreate()

In [0]:
# Pruebas saber 11 https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe/about_data
Pruebas = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/workspace/pdge(saber-11)/saber11/BRONZE/saber.csv")

In [0]:
Pruebas.printSchema()

In [0]:
# Conteo de nulos
Pruebas.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in Pruebas.columns]).show()

In [0]:
string_cols = [c for c, t in Pruebas.dtypes if t == "string"]

for c in string_cols:
    Pruebas = Pruebas.withColumn(c, F.lower(F.col(c)))

In [0]:
# Inconsistencias case-sensitive
for c in Pruebas.columns:
    Pruebas.select(F.count(F.when(F.col(c) == F.lower(F.col(c)),True)).alias(c)).show()

In [0]:
Pruebas = Pruebas.withColumn(
    "COLE_DEPTO_UBICACION",
    F.when(F.col("COLE_DEPTO_UBICACION") == "bogotá", "bogota")
     .otherwise(F.col("COLE_DEPTO_UBICACION"))
)

In [0]:
# Conteo de duplicados
total     = Pruebas.count()
distintos = Pruebas.distinct().count()

print(f"Total filas:     {total}")
print(f"Filas distintas: {distintos}")
print(f"Duplicados:      {total - distintos}")

## Filtros

In [0]:
nuevo = Pruebas.distinct()

In [0]:
# Identificar columnas con valores únicos
for c in nuevo.columns:
    nuevo.groupBy(c).count().show()

In [0]:
nuevo = nuevo.filter(F.lower(F.col("ESTU_PAIS_RESIDE")) == "colombia")
nuevo = nuevo.filter(F.lower(F.col("ESTU_NACIONALIDAD")) == "colombia")

In [0]:
# Lista de columnas de puntajes
cols_puntajes = [
    "PUNT_GLOBAL",
    "PUNT_C_NATURALES",
    "PUNT_INGLES",
    "PUNT_LECTURA_CRITICA",
    "PUNT_MATEMATICAS",
    "PUNT_SOCIALES_CIUDADANAS"
]

# 1. Cast a double
for col_name in cols_puntajes:
    nuevo = nuevo.withColumn(col_name, F.col(col_name).cast("double"))

# 2. Filtrar valores válidos (> 0)
for col_name in cols_puntajes:
    nuevo = nuevo.filter(F.col(col_name) > 0)

In [0]:
# Transformación de fecha necesario para filtrar
nuevo = nuevo.withColumn("ANIO", (F.col("PERIODO") / 10).cast("int")) \
       .withColumn("PERIODO", (F.col("PERIODO") % 10))

# Filtro de años 2015 - 2023
nuevo = nuevo.filter(F.col("ANIO").between(2015, 2023))


In [0]:
cols = ["ESTU_TIPODOCUMENTO","ESTU_CONSECUTIVO","COLE_COD_DANE_SEDE","COLE_CODIGO_ICFES","COLE_NOMBRE_ESTABLECIMIENTO","COLE_NOMBRE_SEDE","COLE_SEDE_PRINCIPAL","ESTU_COD_DEPTO_PRESENTACION","ESTU_COD_MCPIO_PRESENTACION","ESTU_DEPTO_PRESENTACION","ESTU_ESTADOINVESTIGACION","ESTU_FECHANACIMIENTO","ESTU_MCPIO_PRESENTACION","ESTU_NACIONALIDAD","ESTU_PAIS_RESIDE","ESTU_ESTUDIANTE","FAMI_CUARTOSHOGAR","FAMI_PERSONASHOGAR","FAMI_TIENELAVADORA","FAMI_EDUCACIONMADRE","FAMI_EDUCACIONPADRE","ESTU_GENERO","FAMI_ESTRATOVIVIENDA","FAMI_TIENEAUTOMOVIL","FAMI_TIENECOMPUTADOR","FAMI_TIENEINTERNET"]
for c in cols:
    nuevo = nuevo.drop(c)

## Transformaciones

In [0]:
transformada = nuevo

In [0]:
transformada = transformada.dropna()

In [0]:
from pyspark.sql.window import Window

wdpto = (Window.partitionBy("COLE_DEPTO_UBICACION").orderBy(F.desc("PUNT_GLOBAL")))
wmpio = (Window.partitionBy("COLE_MCPIO_UBICACION").orderBy(F.desc("PUNT_GLOBAL")))
wanio = (Window.partitionBy("ANIO").orderBy(F.desc("PUNT_GLOBAL")))

transformada = transformada.withColumn("AVG_DEPTO", F.avg("PUNT_GLOBAL").over(wdpto)) \
                           .withColumn("AVG_MPIO", F.avg("PUNT_GLOBAL").over(wmpio)) \
                           .withColumn("AVG_ANIO", F.avg("PUNT_GLOBAL").over(wanio))

In [0]:
transformada.createOrReplaceTempView("TransformadaView")
spark.sql("""SELECT COLE_DEPTO_UBICACION, AVG(PUNT_GLOBAL)
           FROM TransformadaView
           GROUP BY COLE_DEPTO_UBICACION
           order by AVG(PUNT_GLOBAL) DESC
           LIMIT 10
          """).show()

In [0]:
transformada.createOrReplaceTempView("TransformadaView")
spark.sql("""SELECT COLE_DEPTO_UBICACION, AVG(PUNT_GLOBAL)
           FROM TransformadaView
           GROUP BY COLE_DEPTO_UBICACION
           order by AVG(PUNT_GLOBAL) ASC
           LIMIT 10
          """).show()

In [0]:
transformada.createOrReplaceTempView("TransformadaView")
spark.sql("""SELECT COLE_NATURALEZA, AVG(PUNT_GLOBAL)
           FROM TransformadaView
           GROUP BY COLE_NATURALEZA
           order by AVG(PUNT_GLOBAL) DESC
          """).show()

In [0]:
transformada.display()

In [0]:
transformada.describe().show()

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# 1. Promedio anual por materia
desempeno_anual = (
    transformada
    .groupBy("ANIO")
    .agg(
        F.avg("PUNT_INGLES").alias("PROM_INGLES"),
        F.avg("PUNT_MATEMATICAS").alias("PROM_MATEMATICAS"),
        F.avg("PUNT_SOCIALES_CIUDADANAS").alias("PROM_SOCIALES"),
        F.avg("PUNT_C_NATURALES").alias("PROM_NATURALES"),
        F.avg("PUNT_LECTURA_CRITICA").alias("PROM_LECTURA")
    )
    .orderBy("ANIO")
)

# 2. Pasar a pandas
pdf = desempeno_anual.toPandas()

# 3. Graficar
plt.figure(figsize=(12, 6))

plt.plot(pdf["ANIO"], pdf["PROM_INGLES"], marker="o", label="Ingles")
plt.plot(pdf["ANIO"], pdf["PROM_MATEMATICAS"], marker="o", label="Matematicas")
plt.plot(pdf["ANIO"], pdf["PROM_SOCIALES"], marker="o", label="Sociales y ciudadanas")
plt.plot(pdf["ANIO"], pdf["PROM_NATURALES"], marker="o", label="C. naturales")
plt.plot(pdf["ANIO"], pdf["PROM_LECTURA"], marker="o", label="Lectura critica")

plt.title("Desempeno promedio general por materia y anio")
plt.xlabel("Anio")
plt.ylabel("Puntaje promedio")
plt.xticks(pdf["ANIO"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
print("\nTabla en formato LaTeX:\n")
print(pdf.to_latex(index=False))

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd

# Extraer la columna de interes
pdf_global = transformada.select("PUNT_GLOBAL").dropna().toPandas()

# Histograma
plt.figure(figsize=(10, 6))
plt.hist(pdf_global["PUNT_GLOBAL"], bins=40, edgecolor="black", alpha=0.8)

plt.title("Distribucion del Puntaje Global ICFES Saber 11")
plt.xlabel("Puntaje global")
plt.ylabel("Frecuencia")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()